# 09 · From prediction to the next step
**Kuizi brain walkthrough, notebook 9 of 10**

SAINT+ predicts one thing: the chance of a right answer on a given item, now. Everything the
student sees is built on that:

1. **Mastery per concept** (T049): ask the model about a *typical* item of each concept.
2. **Explanation** (T049, T177): the attention weights say which past answers drove each estimate.
3. **Next step** (T052, T053): if the target concept is weak, walk the graph back to the weakest
   prerequisite (notebook 1); otherwise pick the item with a success chance near 70 %, hard enough
   to learn from, easy enough not to discourage.

<div dir="rtl" lang="ar">

## ٠٩ · من التنبّؤ إلى الخطوة التالية

يتنبّأ SAINT+ بشيء واحد: احتمال الإجابة الصحيحة على تمرين معيّن الآن. وكل ما يراه التلميذ مبنيّ على ذلك:
١. <b>التمكّن في كل مفهوم</b> (T049): نسأل النموذج عن تمرين <b>نموذجي</b> من كل مفهوم.
٢. <b>الشرح</b> (T049، T177): أوزان الانتباه تقول أي الإجابات السابقة أثّرت في كل تقدير.
٣. <b>الخطوة التالية</b> (T052، T053): إذا كان المفهوم المستهدف ضعيفًا، نرجع على الخريطة إلى أضعف مكتسب قبلي (الدفتر ١)؛
وإلا نختار تمرينًا احتمال نجاحه قرابة 70%: صعب بما يكفي ليتعلّم منه، وسهل بما يكفي كي لا يُحبَط.

</div>

In [ ]:
# Concept: model settings · Production: T049
EPOCHS, BATCH, LR = 6, 16, 1e-3   # SAINT+ training settings (see notebook 7)

In [ ]:
# Concept: training SAINT+ on simulated students · Production: T042
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from kuizi_brain import data as D, models as M, metrics as X, graph as G
torch.manual_seed(0)
sim = D.simulate_students(n_students=1500, answers_per_student=120, seed=7)
train, test = D.split_by_student(sim)
saint = M.SaintPlus(sim.attrs["n_items"], sim.attrs["n_kc"])
M.fit(saint, M.make_windows(train), epochs=EPOCHS, batch_size=BATCH, lr=LR, verbose=False)
item_kc, item_b = sim.attrs["item_kc"], sim.attrs["item_difficulty"]
kc_ids = G.topological_order()          # simulator concept index -> graph id
print("trained")

In [ ]:
# Concept: mastery per concept for one student, compared with the truth · Production: T049
student = test.student.unique()[5]
hist = test[test.student == student]
mastery, attention = M.probe_mastery(saint, hist, item_kc, item_b)
truth = sim.attrs["final_p_true"][student]
table = pd.DataFrame({"concept": [G.PHYSICS_KCS[kc_ids[k]][0] for k in mastery],
                      "SAINT+ estimate": [mastery[k] for k in mastery],
                      "true (hidden)": [truth[k] for k in mastery],
                      "answers seen": [int((hist.kc == k).sum()) for k in mastery]}).round(2)
table

In [ ]:
# Concept: mastery on the concept graph · Production: T049, T177
fig, ax = plt.subplots(figsize=(13, 6))
G.draw({kc_ids[k]: v for k, v in mastery.items()}, ax=ax)
ax.set_title("SAINT+ mastery estimates for one simulated student (dashed = below 60 %)")
plt.show()
all_est, all_true = [], []
for s in test.student.unique()[:150]:
    est, _ = M.probe_mastery(saint, test[test.student == s], item_kc, item_b)
    all_est += list(est.values()); all_true += list(sim.attrs["final_p_true"][s])
print(f"Across 150 test students, correlation between estimated and true mastery: {np.corrcoef(all_est, all_true)[0, 1]:.2f}")

In [ ]:
# Concept: explaining one estimate with attention · Production: T049, T177
target = min(mastery, key=lambda k: mastery[k] if (hist.kc == k).any() else 9)
weights = attention[target]
h = hist.tail(len(weights)).reset_index(drop=True)
top = np.argsort(weights)[::-1][:5]
print(f"Why is '{G.PHYSICS_KCS[kc_ids[target]][0]}' estimated at {mastery[target]:.0%}? The answers that weigh most:")
for i in top:
    print(f"  {weights[i]:.2f}  answer #{i + 1}: {h.kc_name[i]} -> {'right' if h.correct[i] else 'wrong'}")

In [ ]:
# Concept: choosing the next step · Production: T052, T053, T056
names = {k: G.PHYSICS_KCS[kc_ids[k]][0] for k in mastery}
last_kc = int(hist.kc.iloc[-1])
target_id = kc_ids[last_kc]
by_id = {kc_ids[k]: v for k, v in mastery.items()}
weak = G.weakest_prerequisite(target_id, by_id, threshold=0.6)
if by_id[target_id] < 0.6 and weak is not None:
    print(f"Target {G.PHYSICS_KCS[target_id][0]} is weak ({by_id[target_id]:.0%}) and so is its prerequisite "
          f"{G.PHYSICS_KCS[weak][0]} ({by_id[weak]:.0%}): 1-minute remediation on {G.PHYSICS_KCS[weak][0]} first.")
    focus = kc_ids.index(weak)
else:
    print(f"Target {G.PHYSICS_KCS[target_id][0]} ({by_id[target_id]:.0%}): practise it directly.")
    focus = last_kc
candidates = np.flatnonzero(item_kc == focus)
p, _ = M.probe(saint, hist, candidates, item_kc)
best = candidates[np.argmin(np.abs(p - 0.7))]
print(f"Next item: #{best} on {names[focus]}, predicted success {p[candidates == best][0]:.0%} "
      f"(the closest to 70 % among {len(candidates)} items; range {p.min():.0%}-{p.max():.0%}).")

### Remember
- Mastery = the model's chance of success on a typical item of each concept, asked "now".
- Attention weights turn a number into an explanation a student, parent or teacher can read.
- The next step combines the brain (probabilities) with the concept graph (where to go back to).
- Targeting about 70 % success keeps practice useful and motivating.

<div dir="rtl" lang="ar">

### تذكّر
- التمكّن = احتمال النجاح الذي يعطيه النموذج لتمرين نموذجي من كل مفهوم، "الآن".
- أوزان الانتباه تحوّل الرقم إلى شرح يقرؤه التلميذ والوليّ والأستاذ.
- الخطوة التالية تجمع بين الدماغ (الاحتمالات) وخريطة المفاهيم (إلى أين نرجع).
- استهداف نحو 70% من النجاح يجعل التمرّن مفيدًا ومحفّزًا.

</div>